# 01 — Exploratory Data Analysis

Reads the cleaned Google Transparency Report data from the S3 `sample/` extract
(200 k requests / 3.6 M domain rows) and answers four questions:

1. What does the `removal_rate` distribution look like, and how bimodal is it?
2. Who files the most requests, and who gets the best outcomes?
3. How has request volume changed over time?
4. How much of the data is near-zero-removal **noise**, and where does it cluster?

All charts use Plotly so they are interactive in the dashboard later.

In [ ]:
import io
import os
import sys

import boto3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from sift.config import load_settings

settings = load_settings()
BUCKET = settings.s3_bucket or 'sift-piracy-data'
ACTION_THRESHOLD = settings.get('label', 'action_threshold', default=0.5)
print(f'bucket={BUCKET}  action_threshold={ACTION_THRESHOLD}')

In [ ]:
def read_parquet_prefix(bucket: str, prefix: str, max_parts: int = 999) -> pd.DataFrame:
    """Download Parquet parts from an S3 prefix and concatenate."""
    s3 = boto3.client('s3')
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    keys = sorted(
        o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')
    )[:max_parts]
    frames = []
    for key in keys:
        obj = s3.get_object(Bucket=bucket, Key=key)
        frames.append(pd.read_parquet(io.BytesIO(obj['Body'].read())))
    return pd.concat(frames, ignore_index=True)

req = read_parquet_prefix(BUCKET, 'sample/requests/')
dom = read_parquet_prefix(BUCKET, 'sample/domains/')

req['date'] = pd.to_datetime(req['date'])
req['year_month'] = req['date'].dt.to_period('M').dt.to_timestamp()

print(f'requests: {len(req):,} rows  |  domains: {len(dom):,} rows')
print(f'date range: {req["date"].min().date()} → {req["date"].max().date()}')

## 1 — Removal-rate distribution

The model's label source. A bimodal distribution (mass near 0 and near 1) is
what we expect: most requests are either fully actioned or fully rejected.

In [ ]:
fig = px.histogram(
    req, x='removal_rate', nbins=50,
    title='Removal-rate distribution (request level)',
    labels={'removal_rate': 'Removal rate', 'count': 'Requests'},
    color_discrete_sequence=['#636EFA'],
)
fig.add_vline(
    x=ACTION_THRESHOLD, line_dash='dash', line_color='red',
    annotation_text=f'Action threshold ({ACTION_THRESHOLD})',
    annotation_position='top right',
)
fig.update_layout(bargap=0.05)
fig.show()

noise_pct = (req['removal_rate'] < ACTION_THRESHOLD).mean() * 100
action_pct = 100 - noise_pct
print(f'Actioned (rate ≥ {ACTION_THRESHOLD}): {action_pct:.1f}%')
print(f'Noise    (rate <  {ACTION_THRESHOLD}): {noise_pct:.1f}%')

## 2 — Top reporting organisations and copyright owners

In [ ]:
top_orgs = (
    req.groupby('reporting_org')
    .agg(requests=('request_id', 'count'), mean_rate=('removal_rate', 'mean'))
    .sort_values('requests', ascending=False)
    .head(20)
    .reset_index()
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Requests filed', 'Mean removal rate'),
)
fig.add_trace(
    go.Bar(x=top_orgs['requests'], y=top_orgs['reporting_org'],
           orientation='h', name='Requests', marker_color='#636EFA'),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=top_orgs['mean_rate'], y=top_orgs['reporting_org'],
           orientation='h', name='Mean rate', marker_color='#EF553B'),
    row=1, col=2,
)
fig.update_layout(title='Top 20 reporting organisations', height=600, showlegend=False)
fig.update_yaxes(autorange='reversed')
fig.show()

In [ ]:
top_owners = (
    req.groupby('copyright_owner')
    .agg(requests=('request_id', 'count'), mean_rate=('removal_rate', 'mean'))
    .sort_values('requests', ascending=False)
    .head(20)
    .reset_index()
)

fig = px.scatter(
    top_owners, x='requests', y='mean_rate',
    text='copyright_owner', size='requests',
    title='Top 20 copyright owners — volume vs. mean removal rate',
    labels={'requests': 'Requests filed', 'mean_rate': 'Mean removal rate'},
    color='mean_rate', color_continuous_scale='RdYlGn',
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_hline(y=ACTION_THRESHOLD, line_dash='dash', line_color='grey')
fig.show()

## 3 — Request volume over time

In [ ]:
monthly = (
    req.groupby('year_month')
    .agg(
        total=('request_id', 'count'),
        actioned=('removal_rate', lambda s: (s >= ACTION_THRESHOLD).sum()),
    )
    .reset_index()
)
monthly['noise'] = monthly['total'] - monthly['actioned']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly['year_month'], y=monthly['actioned'],
    name='Actioned', marker_color='#00CC96',
))
fig.add_trace(go.Bar(
    x=monthly['year_month'], y=monthly['noise'],
    name='Noise (rate < threshold)', marker_color='#EF553B',
))
fig.update_layout(
    barmode='stack',
    title='Monthly request volume — actioned vs. noise',
    xaxis_title='Month', yaxis_title='Requests',
)
fig.show()

## 4 — Quantifying the noise problem

"Noise" = requests where almost nothing was actioned (`removal_rate < threshold`).
This section shows *how much* noise there is and *where* it clusters.

In [ ]:
# Near-zero = removal_rate exactly 0 (nothing actioned at all)
zero_rate = req[req['removal_rate'] == 0.0]
below_thresh = req[req['removal_rate'] < ACTION_THRESHOLD]

print('=== Noise quantification ===')
print(f'  removal_rate == 0.0 : {len(zero_rate):>8,}  ({len(zero_rate)/len(req)*100:.1f}%)')
print(f'  removal_rate < {ACTION_THRESHOLD}  : {len(below_thresh):>8,}  ({len(below_thresh)/len(req)*100:.1f}%)')
print(f'  from_abuser == True : {req["from_abuser"].sum():>8,}  ({req["from_abuser"].mean()*100:.2f}%)')

In [ ]:
# Which orgs produce the most noise?
org_noise = (
    req.groupby('reporting_org')
    .agg(
        total=('request_id', 'count'),
        noise=('removal_rate', lambda s: (s < ACTION_THRESHOLD).sum()),
    )
    .assign(noise_pct=lambda d: d['noise'] / d['total'])
    .query('total >= 50')  # ignore orgs with tiny sample
    .sort_values('noise_pct', ascending=False)
    .head(15)
    .reset_index()
)

fig = px.bar(
    org_noise, x='noise_pct', y='reporting_org', orientation='h',
    color='noise_pct', color_continuous_scale='Reds',
    title='Top 15 organisations by noise rate (removal_rate < threshold)',
    labels={'noise_pct': 'Noise rate', 'reporting_org': 'Reporting org'},
    text=org_noise['total'].apply(lambda n: f'n={n:,}'),
)
fig.update_layout(yaxis={'autorange': 'reversed'}, coloraxis_showscale=False)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Domain-level noise: TLDs most associated with zero-removal domains
dom_with_tld = dom.copy()
dom_with_tld['tld'] = dom_with_tld['domain'].str.extract(r'\.([a-z]{2,})$', expand=False).fillna('other')

tld_noise = (
    dom_with_tld.groupby('tld')
    .agg(
        domains=('domain', 'count'),
        noise=('removal_rate', lambda s: (s < ACTION_THRESHOLD).sum()),
    )
    .assign(noise_pct=lambda d: d['noise'] / d['domains'])
    .query('domains >= 500')
    .sort_values('noise_pct', ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    tld_noise, x='tld', y='noise_pct',
    color='noise_pct', color_continuous_scale='Oranges',
    title='Noise rate by TLD (domains with ≥ 500 appearances)',
    labels={'noise_pct': 'Noise rate', 'tld': 'TLD'},
    text=tld_noise['domains'].apply(lambda n: f'{n:,}'),
)
fig.update_layout(coloraxis_showscale=False)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Suspicious-token domains: do they have higher removal rates (as expected)?
from sift.features.build_features import DEFAULT_SUSPICIOUS_TOKENS
import re

token_pattern = '|'.join(re.escape(t) for t in DEFAULT_SUSPICIOUS_TOKENS)
dom['has_token'] = dom['domain'].fillna('').str.contains(token_pattern, regex=True)

token_summary = dom.groupby('has_token')['removal_rate'].describe()[['count','mean','50%']]
token_summary.index = ['No piracy token', 'Has piracy token']
print('Removal rate by suspicious-token presence:')
print(token_summary.to_string())

fig = px.box(
    dom.sample(min(50_000, len(dom)), random_state=42),
    x='has_token', y='removal_rate',
    title='Removal rate: domains with vs. without suspicious tokens',
    labels={'has_token': 'Contains suspicious token', 'removal_rate': 'Removal rate'},
    color='has_token',
)
fig.show()

## 5 — Summary findings

Key numbers to carry into the modelling phase:

In [ ]:
print('=== Summary ===')
print(f'Total requests in sample : {len(req):>10,}')
print(f'Total domains  in sample : {len(dom):>10,}')
print()
print(f'Actioned (rate >= {ACTION_THRESHOLD})     : {(req.removal_rate >= ACTION_THRESHOLD).mean()*100:>6.1f}%')
print(f'Noise    (rate <  {ACTION_THRESHOLD})     : {(req.removal_rate <  ACTION_THRESHOLD).mean()*100:>6.1f}%')
print(f'Zero-removal requests    : {(req.removal_rate == 0).mean()*100:>6.1f}%')
print()
print(f'Unique reporting orgs    : {req.reporting_org.nunique():>10,}')
print(f'Unique copyright owners  : {req.copyright_owner.nunique():>10,}')
print()
print(f'Date range               : {req.date.min().date()} → {req.date.max().date()}')